# Quantum Funding Geographic Bubble Chart | Smooth World Edition

This version uses a **smooth gray world silhouette** as a reusable Matplotlib base layer. The old low-poly facet texture has been removed. Regional halos, country bubbles, annotations, funding notes, totals, and signature remain separate chart layers.

Run all cells from the top in a fresh Colab runtime. The world base is defined in **3A. Reusable smooth world layer** and drawn by `draw_world_low_poly(ax)` for compatibility with the plotting cell.

Data note: China is styled like other country bubbles, but its cumulative estimate is flagged in the notes and excluded from annual totals. This is an editorial geographic illustration, not a GIS basemap.

In [ ]:
from pathlib import Path
import os
import subprocess

PROJECT_DIR = Path("/content/mit-prox-rd")
REPO_URL = "https://github.com/qimono-repos/mit-prox-rd.git"

# Clone trunk on first run; refresh it on later runs.
if not (PROJECT_DIR / ".git").exists():
    subprocess.run(
        ["git", "clone", "--branch", "trunk", "--single-branch", REPO_URL, str(PROJECT_DIR)],
        check=True,
    )
else:
    subprocess.run(["git", "-C", str(PROJECT_DIR), "fetch", "origin", "trunk"], check=True)
    subprocess.run(["git", "-C", str(PROJECT_DIR), "checkout", "-B", "trunk", "FETCH_HEAD"], check=True)

os.chdir(PROJECT_DIR)
print("Project directory:", Path.cwd())
print("CSV exists:", Path("assets/quantum_funding_updated.csv").exists())

## 1. Imports and Data Loading

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, Polygon
import numpy as np
from pathlib import Path

csv_path = Path("assets/quantum_funding_updated.csv")
if not csv_path.is_file():
    raise FileNotFoundError(f"CSV not found: {csv_path.resolve()}")

df = pd.read_csv(csv_path, encoding="utf-8-sig")
expected_columns = {"Rank", "Country / Entity", "highest confidence figure USD Millions", "Type", "ssource-link", "observations"}
missing = expected_columns.difference(df.columns)
if missing:
    raise ValueError(f"CSV is missing expected columns: {sorted(missing)}")

print(f"Loaded {len(df)} records and {len(df.columns)} columns from {csv_path}.")
display(df.head())

## 2. Configuration

Define the values used for bubble sizing and the approximate geographic coordinates.

In [ ]:
values = {
    "United States": 1006,
    "Canada": 38.5,
    "Brazil": 5.5,
    "United Kingdom": 330,
    "Germany": 848,
    "Netherlands": 85,
    "Switzerland": 23,
    "Austria": 24,
    "Italy": 36,
    "India": 90,
    "South Korea": 140,
    "Japan": 241,
    "Singapore": 44,
    "Australia": None,
    "China": 15000,
}

pos = {
    "Canada": (-105, 57),
    "United States": (-99, 37),
    "Brazil": (-52, -10),
    "United Kingdom": (-3, 54),
    "Netherlands": (5, 53),
    "Germany": (11, 51),
    "Switzerland": (8, 47),
    "Austria": (14, 47),
    "Italy": (12, 42),
    "India": (79, 22),
    "South Korea": (128, 36),
    "Japan": (139, 37),
    "Singapore": (104, 1.3),
    "Australia": (134, -25),
    "China": (105, 35),
}

## 3. Generate the Chart

### 3A. Reusable low-poly world layer

This cell owns the world silhouette independently from the funding bubbles. The coastlines are hand-drawn as geographic polygon outlines, then filled with subtle triangular facets to create a quiet gray low-poly atlas. Reuse `draw_world_low_poly(ax)` in other Matplotlib charts by copying this cell and calling the function before plotting data. No image files, SVG assets, Cartopy, GeoPandas, or Shapely are required.

In [ ]:
# REUSABLE WORLD LAYER | pure Matplotlib
# A smooth, recognizable world silhouette. No low-poly facets.
# Coordinates are longitude/latitude. This is an editorial illustration, not a GIS basemap.

from matplotlib.patches import Polygon

WORLD_FACE = "#e3e6e9"
WORLD_EDGE = "#cbd2d8"

# Smooth, recognizable coastline outlines. Each polygon represents a major landmass;
# avoid overlapping country-sized patches or interior triangles.
WORLD_CONTINENTS = {
    "north_america": [
        (-168,66),(-162,70),(-153,70),(-146,66),(-139,61),(-132,58),(-128,52),
        (-125,48),(-123,42),(-117,33),(-111,30),(-105,26),(-98,22),(-91,18),
        (-86,12),(-82,8),(-78,8),(-80,15),(-76,22),(-80,28),(-76,34),
        (-70,42),(-64,47),(-59,53),(-61,58),(-72,62),(-85,65),(-99,70),
        (-117,73),(-135,72),(-151,70),(-161,68)
    ],
    "greenland": [
        (-53,83),(-44,84),(-34,81),(-25,76),(-29,70),(-38,64),(-47,59),
        (-55,61),(-61,68),(-59,76)
    ],
    "central_america": [
        (-96,20),(-91,18),(-87,15),(-84,11),(-81,8),(-78,9),(-79,13),
        (-83,17),(-88,20)
    ],
    "south_america": [
        (-81,12),(-76,9),(-71,7),(-65,3),(-59,0),(-52,-4),(-46,-9),
        (-40,-16),(-42,-23),(-47,-31),(-52,-39),(-57,-47),(-63,-55),
        (-68,-51),(-71,-41),(-74,-28),(-78,-13),(-80,0)
    ],
    "eurasia": [
        (-10,36),(-9,43),(-5,49),(1,53),(7,57),(13,60),(20,62),(27,67),
        (37,70),(47,67),(56,64),(66,62),(76,60),(87,58),(98,56),(110,53),
        (122,51),(135,49),(148,46),(160,42),(169,35),(166,29),(157,23),
        (148,18),(140,11),(132,6),(124,8),(117,14),(109,19),(101,22),
        (93,20),(85,24),(77,28),(69,33),(60,38),(52,37),(45,33),(38,29),
        (32,31),(27,37),(20,39),(14,36),(8,36),(3,40),(-2,39)
    ],
    "africa": [
        (-17,36),(-5,37),(8,35),(19,32),(28,27),(34,19),(38,9),(43,11),
        (50,11),(49,3),(42,-2),(38,-12),(32,-22),(25,-33),(17,-35),
        (10,-29),(4,-18),(-1,-5),(-7,8),(-13,20)
    ],
    "arabia": [(35,30),(47,28),(56,19),(51,13),(43,12),(39,20)],
    "india": [(68,27),(78,30),(88,23),(84,13),(77,7),(73,16)],
    "southeast_asia": [
        (93,20),(104,20),(114,14),(121,6),(116,-4),(109,-8),(103,0),(98,9)
    ],
    "japan": [(130,45),(136,43),(142,39),(145,35),(141,32),(136,34),(132,39)],
    "british_isles": [(-8,59),(-4,58),(-2,54),(-4,51),(-7,50),(-10,54)],
    "madagascar": [(49,-12),(51,-17),(49,-25),(46,-21)],
    "indonesia": [(96,5),(106,2),(117,-4),(128,-5),(135,-8),(124,-10),(113,-7),(104,-5)],
    "australia": [
        (113,-11),(115,-19),(121,-24),(129,-25),(137,-17),(145,-15),
        (153,-25),(149,-37),(139,-43),(128,-40),(117,-34),(114,-24)
    ],
    "new_zealand": [(172,-35),(178,-41),(174,-46),(169,-42)],
}

def _smooth_closed_coastline(points, iterations=3):
    """Smooth closed coastline coordinates using Chaikin subdivision."""
    pts = [(float(x), float(y)) for x, y in points]
    for _ in range(iterations):
        refined = []
        for i, p in enumerate(pts):
            q = pts[(i + 1) % len(pts)]
            refined.append((0.75*p[0] + 0.25*q[0], 0.75*p[1] + 0.25*q[1]))
            refined.append((0.25*p[0] + 0.75*q[0], 0.25*p[1] + 0.75*q[1]))
        pts = refined
    return pts

def draw_world_low_poly(ax, *, zorder=1, show_facets=False):
    """Draw a reusable smooth gray world silhouette on a Matplotlib axis.

    The legacy function name is retained for compatibility. The map has no facets:
    smooth coastline shapes form a quiet base layer behind regional bubbles.
    """
    islands = {"greenland", "british_isles", "madagascar", "new_zealand", "japan"}
    for name, points in WORLD_CONTINENTS.items():
        smooth = _smooth_closed_coastline(points, iterations=3)
        patch = Polygon(
            smooth, closed=True,
            facecolor="#e8ebee" if name in islands else WORLD_FACE,
            edgecolor=WORLD_EDGE, linewidth=0.65 if name in islands else 0.8,
            joinstyle="round", antialiased=True, zorder=zorder
        )
        ax.add_patch(patch)
    return ax


In [ ]:
# Publication-style geographic funding infographic, pure Matplotlib (no Cartopy/GeoPandas).
# Country figures are loaded from the CSV. China uses the same bubble style as other countries;
# its cumulative estimate is explicitly caveated in the notes and is excluded from annual totals.

import math
from matplotlib.patches import Circle, Polygon, Ellipse
from matplotlib.lines import Line2D

# --- Data selection and region membership ---
name_col = "Country / Entity"
value_col = "highest confidence figure USD Millions"
type_col = "Type"
obs_col = "observations"

data = df.copy()
data[value_col] = pd.to_numeric(data[value_col], errors="coerce")

# The US has separate FY2023/FY2024/FY2025 rows: use FY2024 enacted for this snapshot.
us_rows = data[data[name_col].eq("United States")]
us_2024 = us_rows[us_rows[type_col].astype(str).str.contains("FY2024 enacted", case=False, na=False)]
selected = data[~data[name_col].eq("United States")].copy()
if not us_2024.empty:
    selected = pd.concat([selected, us_2024.iloc[[0]]], ignore_index=True)
else:
    selected = pd.concat([selected, us_rows.iloc[[0]]], ignore_index=True)

# Keep country rows only; regional/cumulative aggregates must not be counted as countries.
excluded_entities = {
    "Confirmed global total", "Europe (confirmed)", "Asia-Pacific (confirmed)",
    "Americas (confirmed)", "European Union (Flagship)"
}
selected = selected[~selected[name_col].isin(excluded_entities)]
country_values = dict(zip(selected[name_col], selected[value_col]))

regions = {
    "Americas": ["United States", "Canada", "Brazil"],
    "Europe": ["United Kingdom", "Germany", "Netherlands", "Switzerland", "Austria", "Italy"],
    "Asia": ["Russia", "China", "India", "Japan", "South Korea"],
    "APAC": ["Singapore", "Australia"],
}
region_colors = {
    "Americas": "#159bd7",
    "Europe": "#f04462",
    "Asia": "#f28b20",
    "APAC": "#8050d7",
}
positions = {
    "Canada": (-105, 56), "United States": (-99, 37), "Brazil": (-52, -10),
    "United Kingdom": (-3, 54), "Netherlands": (5, 53), "Germany": (11, 51),
    "Switzerland": (8, 47), "Austria": (14, 47), "Italy": (12, 42),
    "Russia": (90, 57), "India": (79, 22), "China": (105, 35),
    "Japan": (139, 37), "South Korea": (128, 36),
    "Singapore": (104, 1.3), "Australia": (134, -25),
}
# Label offsets keep dense European labels legible and make missing data explicit.
label_offsets = {
    "United States": (-22, 3), "Canada": (12, 7), "Brazil": (10, -2),
    "United Kingdom": (-22, 9), "Netherlands": (-12, 14), "Germany": (13, 8),
    "Switzerland": (13, -3), "Austria": (13, -10), "Italy": (-13, -11),
    "Russia": (9, 8), "India": (5, -10), "China": (11, 0),
    "Japan": (8, 7), "South Korea": (11, -6), "Singapore": (8, 0),
    "Australia": (11, 0),
}

def fmt_usd_millions(value):
    if pd.isna(value):
        return "no comparable total"
    if value >= 1000:
        return f"${value/1000:.2f}B"
    return f"${value:.0f}M"

def region_total(region):
    # China is cumulative and Australia has no comparable dedicated total.
    members = [n for n in regions[region] if n != "China" and not pd.isna(country_values.get(n, np.nan))]
    return sum(float(country_values[n]) for n in members)

# --- Canvas and editorial palette ---
fig = plt.figure(figsize=(19, 12), dpi=170, facecolor="#fbfaf7")
ax = fig.add_axes([0.035, 0.19, 0.93, 0.63])
ax.set_facecolor("#fbfaf7")
ink, muted = "#111b3c", "#59647b"
ax.set_xlim(-178, 185)
ax.set_ylim(-59, 83)
ax.axis("off")

# Soft coordinate grid
for lon in [-120, -60, 0, 60, 120]:
    ax.axvline(lon, color="#e8e9ee", lw=0.65, zorder=0)
for lat in [-30, 0, 30, 60]:
    ax.axhline(lat, color="#e8e9ee", lw=0.65, zorder=0)

# Reusable world layer: see section 3A for the low-poly geometry and drawing function.
draw_world_low_poly(ax)

# Translucent regional halos provide the large "continent bubble" layer.
region_halos = {
    "Americas": (-104, 30, 76, 93),
    "Europe": (17, 43, 64, 67),
    "Asia": (94, 36, 104, 78),
    "APAC": (132, -19, 63, 51),
}
for region, (cx, cy, w, h) in region_halos.items():
    ax.add_patch(Ellipse((cx, cy), width=w, height=h,
                         facecolor=region_colors[region], edgecolor=region_colors[region],
                         linewidth=1.4, alpha=0.105, zorder=1.5))

# Region headings and annual/annualized totals. China is deliberately excluded.
region_heading_positions = {
    "Americas": (-159, 66), "Europe": (-29, 73), "Asia": (111, 72), "APAC": (151, 0)
}
for region, (tx, ty) in region_heading_positions.items():
    total = region_total(region)
    ax.text(tx, ty, region, color=region_colors[region], fontsize=17,
            fontweight="bold", ha="left", va="center", zorder=7)
    ax.text(tx, ty-5.1, fmt_usd_millions(total), color=region_colors[region],
            fontsize=15, fontweight="bold", ha="left", va="center", zorder=7)
    ax.text(tx, ty-8.6, "listed countries" + (", excl. China" if region == "Asia" else ""),
            color=muted, fontsize=8.5, ha="left", va="center", zorder=7)

# Country bubbles use one consistent visual language, including China.
available_values = [float(v) for n, v in country_values.items()
                    if n in positions and not pd.isna(v) and n != "China"]
max_value = max(available_values) if available_values else 1
for region, countries in regions.items():
    for country in countries:
        if country not in positions:
            continue
        x, y = positions[country]
        v = country_values.get(country, np.nan)
        color = region_colors[region]
        if pd.isna(v):
            ax.scatter([x], [y], s=110, facecolors="none", edgecolors=color,
                       linewidths=1.7, zorder=5)
            dx, dy = label_offsets[country]
            ax.annotate(f"{country}\nNo comparable total recorded", (x, y),
                        xytext=(x+dx, y+dy), textcoords="data", fontsize=8.2,
                        color=ink, ha="left" if dx >= 0 else "right", va="center",
                        arrowprops=dict(arrowstyle="-", color=color, lw=0.9),
                        zorder=8)
            continue
        # Bubble area scales with value; sqrt radius gives area proportional to value.
        # China is displayed using the same rule but capped visually so it does not
        # swallow the map; its true cumulative amount is explained in the notes.
        if country == "China":
            radius = 8.3
        else:
            radius = 1.5 + 7.6 * math.sqrt(float(v) / max_value)
        ax.add_patch(Circle((x, y), radius=radius, facecolor=color,
                            edgecolor=color, linewidth=1.4, alpha=0.76, zorder=4))
        dx, dy = label_offsets[country]
        label_value = "≈$15B cumulative*" if country == "China" else fmt_usd_millions(float(v))
        # Labels for small/clustered bubbles use leader lines for legibility.
        if country in {"United Kingdom", "Netherlands", "Germany", "Switzerland", "Austria", "Italy",
                       "Canada", "Brazil", "Russia", "India", "South Korea", "Singapore", "Australia"}:
            ax.annotate(f"{country}\n{label_value}", (x, y), xytext=(x+dx, y+dy),
                        textcoords="data", fontsize=8.7, color=ink,
                        fontweight="bold", ha="left" if dx >= 0 else "right", va="center",
                        arrowprops=dict(arrowstyle="-", color=color, lw=0.9,
                                        shrinkA=1, shrinkB=2), zorder=8)
        else:
            ax.annotate(f"{country}\n{label_value}", (x, y), xytext=(x+dx, y+dy),
                        textcoords="data", fontsize=9.4, color=ink,
                        fontweight="bold", ha="left" if dx >= 0 else "right", va="center",
                        arrowprops=dict(arrowstyle="-", color=color, lw=1.0,
                                        shrinkA=1, shrinkB=2), zorder=8)

# --- Header, scale key, totals, regional legend, source and caveats ---
fig.text(0.045, 0.945, "ESTIMATED PUBLIC FUNDING FOR QUANTUM TECHNOLOGIES",
         fontsize=23, fontweight="bold", color=ink, ha="left")
fig.text(0.045, 0.908, "2026 snapshot", fontsize=20, fontweight="bold",
         color=ink, ha="left")
fig.text(0.045, 0.878, "Estimated annual or annualized public funding (USD millions)",
         fontsize=12.5, color=muted, ha="left")
fig.text(0.045, 0.855, "Multi-year budgets annualized where possible; figures are approximate.",
         fontsize=10, color=muted, ha="left")

# Bubble-size key, using illustrative annual funding values.
legend_ax = fig.add_axes([0.79, 0.845, 0.17, 0.12])
legend_ax.set_xlim(0, 10); legend_ax.set_ylim(0, 10); legend_ax.axis("off")
legend_ax.text(0.0, 9.5, "Bubble size guide", fontsize=9, color=muted, weight="bold")
for r, label, yy in [(2.25, "$1B", 6.6), (1.55, "$250M", 3.6), (0.85, "$50M", 1.3)]:
    legend_ax.add_patch(Circle((3.0, yy), r, facecolor="none", edgecolor="#7d8798", lw=0.9))
    legend_ax.text(6.0, yy, label, fontsize=8.5, color=muted, va="center")

# World total is the sum of listed annual/annualized country figures, excluding China.
listed_annual_total = sum(
    float(country_values[n]) for region, members in regions.items() for n in members
    if n != "China" and not pd.isna(country_values.get(n, np.nan))
)
fig.text(0.045, 0.145, "WORLD TOTAL", fontsize=10, color=muted, weight="bold")
fig.text(0.045, 0.108, f"≈ ${listed_annual_total/1000:.2f}B per year",
         fontsize=22, color=ink, weight="bold")
fig.text(0.045, 0.083, "Sum of listed annual / annualized country figures; China excluded.",
         fontsize=8.5, color=muted)

# Region legend doubles as a quick comparison of the regional subtotals.
legend_y = 0.145
legend_xs = [0.36, 0.49, 0.62, 0.75]
for (region, color), x in zip(region_colors.items(), legend_xs):
    fig.patches.append(Circle((x, legend_y), radius=0.008, transform=fig.transFigure,
                              facecolor=color, edgecolor=color, alpha=0.85))
    fig.text(x+0.014, legend_y+0.006, region, fontsize=9, color=ink, weight="bold",
             ha="left", va="center")
    fig.text(x+0.014, legend_y-0.018, fmt_usd_millions(region_total(region)),
             fontsize=9, color=color, weight="bold", ha="left", va="center")

fig.text(0.045, 0.045,
         "Sources: U.S. NQI Annual Report FY2025; McKinsey quantum landscape; national strategy and ministry sources; The Quantum Insider.",
         fontsize=7.8, color=muted, ha="left")
fig.text(0.045, 0.027,
         "Notes: multi-year commitments are annualized estimates, not necessarily annual appropriations or actual expenditure; regional sums include only listed country rows.",
         fontsize=7.4, color=muted, ha="left")
fig.text(0.58, 0.027,
         "China: ~US$15B cumulative estimate through 2024, high uncertainty; not comparable to annual figures and excluded from totals.",
         fontsize=7.4, color=muted, ha="left")
fig.text(0.955, 0.073, "Albert Gruning", fontsize=17, color="#242b3b",
         fontstyle="italic", ha="right")
fig.text(0.955, 0.052, "R&D tech specialist at Qimono (TM)", fontsize=7.7,
         color=muted, ha="right")
fig.text(0.955, 0.035, "Original report for MIT", fontsize=7.7,
         color=muted, ha="right")

plt.show()


## 4. Notes

- U.S. figure is FY2024 enacted.
- Multi-year programs are annualized estimates.
- China is a cumulative estimate with high uncertainty and is not on the same scale.
- See `REFERENCES.md` for full source list.